# Alles Samen - Een Complete Neural Network Library

**Mathematical Foundations - IT & Artificial Intelligence**

---

## 10.0 Recap: De Reis Tot Nu Toe

We hebben een indrukwekkende reis gemaakt door de wiskunde achter deep learning:

### Deel 1: Lineaire Algebra
- **Vectoren en matrices**: data representatie
- **Matrixvermenigvuldiging**: de forward pass
- **Lineaire transformaties**: wat lagen doen

### Deel 2: Calculus
- **Afgeleiden**: hoe verandering te meten
- **Gradient descent**: parameters optimaliseren
- **Backpropagation**: efficiënt gradiënten berekenen

### Deel 3: Statistiek
- **Kansverdelingen**: output interpretatie
- **Verwachtingswaarde/variantie**: weight initialisatie

Nu is het tijd om alles samen te brengen in een **complete, werkende neural network library**!

## 10.1 Leerdoelen

Na deze les kun je een complete neural network library from scratch bouwen. Je begrijpt hoe alle wiskundige concepten samenwerken. Je kunt een flexibel netwerk ontwerpen met verschillende lagen. Je kunt het netwerk trainen en evalueren op echte data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from abc import ABC, abstractmethod

np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)

print("Libraries geladen!")

## 10.2 De Architectuur

Onze library zal bestaan uit:

1. **Layer** (abstracte klasse): interface voor alle lagen
2. **Linear**: lineaire transformatie (Les 2-4)
3. **Activations**: ReLU, Sigmoid, Tanh, Softmax (Les 5)
4. **Loss functions**: MSE, CrossEntropy
5. **Optimizers**: SGD (Les 6)
6. **Sequential**: container voor het netwerk

Dit is vergelijkbaar met hoe PyTorch en Keras werken!

## 10.3 Layer Base Class

Elke laag in ons netwerk moet twee operaties ondersteunen: een **forward pass** (input → output) en een **backward pass** (gradiënt doorsturen). We definiëren dit als een abstracte klasse met `forward()` en `backward()`, net zoals we bij backpropagation (les 7) leerden dat elke schakel in de kettingregel deze twee richtingen heeft.

In [ ]:
class Layer(ABC):
    """Abstracte base class voor alle lagen."""
    
    def __init__(self):
        self.params = {}  # Leerbare parameters
        self.grads = {}   # Gradiënten van parameters
        self.training = True
    
    @abstractmethod
    def forward(self, x):
        """Forward pass: bereken output gegeven input."""
        pass
    
    @abstractmethod
    def backward(self, dout):
        """Backward pass: bereken gradiënten."""
        pass
    
    def __call__(self, x):
        return self.forward(x)
    
    def train(self):
        self.training = True
    
    def eval(self):
        self.training = False

## 10.4 Linear Layer

De kern van neurale netwerken: $z = Wx + b$. Dit is de matrixvermenigvuldiging uit les 2-4.

**Forward** (les 2-3): $z = X W + b$

**Backward** (les 7 — backpropagation):
- $\frac{\partial L}{\partial W} = X^T \cdot \delta_{\text{out}}$
- $\frac{\partial L}{\partial b} = \sum \delta_{\text{out}}$
- $\frac{\partial L}{\partial X} = \delta_{\text{out}} \cdot W^T$

De weights worden geïnitialiseerd met **He-initialisatie** (les 9): $W \sim \mathcal{N}(0, \frac{2}{n_{\text{in}}})$, zodat de variantie van activaties stabiel blijft doorheen de lagen. We testen de laag hieronder met een kleine input van 4 samples en 3 features.

In [ ]:
class Linear(Layer):
    """Lineaire laag: z = Wx + b"""
    
    def __init__(self, in_features, out_features, init='he'):
        super().__init__()
        
        # Initialisatie (Les 9)
        if init == 'he':
            std = np.sqrt(2.0 / in_features)
        elif init == 'xavier':
            std = np.sqrt(2.0 / (in_features + out_features))
        else:
            std = 0.01
        
        self.params['W'] = np.random.randn(in_features, out_features) * std
        self.params['b'] = np.zeros(out_features)
    
    def forward(self, x):
        self.x = x  # Cache voor backward
        return x @ self.params['W'] + self.params['b']
    
    def backward(self, dout):
        n = self.x.shape[0]
        
        # Gradiënten voor parameters
        self.grads['W'] = self.x.T @ dout / n
        self.grads['b'] = np.mean(dout, axis=0)
        
        # Gradiënt voor input (naar vorige laag)
        return dout @ self.params['W'].T

# Test
linear = Linear(3, 2)
x = np.random.randn(4, 3)  # 4 samples, 3 features
out = linear.forward(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {out.shape}")
print(f"W shape: {linear.params['W'].shape}")

## 10.5 Activation Functions

Activatiefuncties introduceren non-lineariteit — zonder hen zou een netwerk met meerdere lagen equivalent zijn aan één enkele lineaire transformatie (les 3). In les 5 leerden we dat de keuze van activatiefunctie invloed heeft op de gradiënt en dus op de training.

Hieronder implementeren we vier activatiefuncties, elk met hun `forward()` en `backward()`. De afgeleide (les 5-7) is cruciaal voor backpropagation: zo stroomt de gradiënt terug door de niet-lineaire lagen.

In [ ]:
class ReLU(Layer):
    """ReLU: f(x) = max(0, x)"""
    
    def forward(self, x):
        self.mask = (x > 0)
        return np.maximum(0, x)
    
    def backward(self, dout):
        return dout * self.mask


class Sigmoid(Layer):
    """Sigmoid: f(x) = 1 / (1 + e^-x)"""
    
    def forward(self, x):
        self.out = 1 / (1 + np.exp(-np.clip(x, -500, 500)))
        return self.out
    
    def backward(self, dout):
        return dout * self.out * (1 - self.out)


class Tanh(Layer):
    """Tanh: f(x) = tanh(x)"""
    
    def forward(self, x):
        self.out = np.tanh(x)
        return self.out
    
    def backward(self, dout):
        return dout * (1 - self.out ** 2)


class LeakyReLU(Layer):
    """Leaky ReLU: f(x) = x if x > 0 else alpha * x"""
    
    def __init__(self, alpha=0.01):
        super().__init__()
        self.alpha = alpha
    
    def forward(self, x):
        self.x = x
        return np.where(x > 0, x, self.alpha * x)
    
    def backward(self, dout):
        return dout * np.where(self.x > 0, 1, self.alpha)

We plotten hieronder elke activatiefunctie $f(x)$ samen met haar afgeleide $f'(x)$. Merk op hoe de afgeleide van Sigmoid en Tanh naar 0 gaat voor grote $|x|$ (het **vanishing gradient** probleem uit les 7), terwijl ReLU een constante gradiënt heeft voor $x > 0$.

In [ ]:
# Visualiseer activatiefuncties
x = np.linspace(-3, 3, 100)

activations = {
    'ReLU': ReLU(),
    'Sigmoid': Sigmoid(),
    'Tanh': Tanh(),
    'LeakyReLU': LeakyReLU(0.1)
}

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, (name, act) in zip(axes.flatten(), activations.items()):
    y = act.forward(x)
    ax.plot(x, y, 'b-', linewidth=2, label='f(x)')
    
    # Toon ook de afgeleide
    dy = act.backward(np.ones_like(x))
    ax.plot(x, dy, 'r--', linewidth=2, label="f'(x)")
    
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(name)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)

plt.tight_layout()
plt.show()

## 10.6 Loss Functions

Loss functies meten hoe goed de voorspellingen van ons model zijn. We implementeren hier drie loss functies:

- **MSE Loss**: voor regressie — equivalent aan de negative log-likelihood onder een Gaussische aanname: $L = \frac{1}{n}\sum_i(y_{\text{pred}} - y_{\text{true}})^2$
- **Cross-Entropy Loss**: voor multi-class classificatie: $L = -\frac{1}{n}\sum_i \log P(y_i \mid x_i)$
- **Binary Cross-Entropy Loss**: voor binaire classificatie: $L = -\frac{1}{n}\sum_i [y_i \log(p_i) + (1-y_i)\log(1-p_i)]$

De `CrossEntropyLoss` combineert softmax (les 8) met de negative log-likelihood in één stap, wat numeriek stabieler is en de elegante gradiënt $\nabla_z L = \hat{y} - y$ oplevert.

In [ ]:
class Loss(ABC):
    """Abstracte base class voor loss functies."""
    
    @abstractmethod
    def forward(self, y_pred, y_true):
        pass
    
    @abstractmethod
    def backward(self):
        pass
    
    def __call__(self, y_pred, y_true):
        return self.forward(y_pred, y_true)


class MSELoss(Loss):
    """Mean Squared Error: L = (1/n) Σ (y_pred - y_true)²"""
    
    def forward(self, y_pred, y_true):
        self.y_pred = y_pred
        self.y_true = y_true
        return np.mean((y_pred - y_true) ** 2)
    
    def backward(self):
        n = self.y_pred.shape[0]
        return 2 * (self.y_pred - self.y_true) / n


class CrossEntropyLoss(Loss):
    """Cross-Entropy Loss met ingebouwde softmax."""
    
    def forward(self, logits, y_true):
        self.y_true = y_true
        n = logits.shape[0]
        
        # Softmax
        exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
        self.probs = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
        
        # Cross-entropy
        correct_logprobs = -np.log(self.probs[np.arange(n), y_true] + 1e-10)
        return np.mean(correct_logprobs)
    
    def backward(self):
        n = self.probs.shape[0]
        grad = self.probs.copy()
        grad[np.arange(n), self.y_true] -= 1
        return grad / n


class BinaryCrossEntropyLoss(Loss):
    """Binary Cross-Entropy Loss."""
    
    def forward(self, y_pred, y_true):
        self.y_pred = np.clip(y_pred, 1e-10, 1 - 1e-10)
        self.y_true = y_true
        return -np.mean(y_true * np.log(self.y_pred) + (1 - y_true) * np.log(1 - self.y_pred))
    
    def backward(self):
        n = self.y_pred.shape[0]
        return (self.y_pred - self.y_true) / (self.y_pred * (1 - self.y_pred) * n)

## 10.7 Optimizers

In les 6 leerden we gradient descent: $\theta \leftarrow \theta - \eta \nabla L$. Hier implementeren we **SGD** (Stochastic Gradient Descent), eventueel met **momentum** (les 6) dat een voortschrijdend gemiddelde van gradiënten bijhoudt om sneller te convergeren en lokale minima te ontwijken.

In [ ]:
class Optimizer(ABC):
    """Abstracte base class voor optimizers."""
    
    def __init__(self, layers, lr=0.01):
        self.layers = layers
        self.lr = lr
    
    @abstractmethod
    def step(self):
        pass
    
    def zero_grad(self):
        for layer in self.layers:
            layer.grads = {}


class SGD(Optimizer):
    """Stochastic Gradient Descent met optioneel momentum."""
    
    def __init__(self, layers, lr=0.01, momentum=0):
        super().__init__(layers, lr)
        self.momentum = momentum
        self.velocity = {}
    
    def step(self):
        for i, layer in enumerate(self.layers):
            for name, param in layer.params.items():
                if name not in layer.grads:
                    continue
                    
                key = (i, name)
                grad = layer.grads[name]
                
                # Momentum
                if self.momentum > 0:
                    if key not in self.velocity:
                        self.velocity[key] = np.zeros_like(param)
                    self.velocity[key] = self.momentum * self.velocity[key] - self.lr * grad
                    layer.params[name] += self.velocity[key]
                else:
                    layer.params[name] -= self.lr * grad

## 10.8 Sequential Model

De `Sequential`-klasse bindt alles samen: het stapelt lagen op in volgorde en voert de forward pass uit door de input achtereenvolgens door elke laag te sturen. Bij de backward pass (les 7) wordt de gradiënt in omgekeerde volgorde doorgestuurd — precies de kettingregel in actie.

In [ ]:
class Sequential:
    """Container voor sequentiële lagen."""
    
    def __init__(self, layers):
        self.layers = layers
    
    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x
    
    def backward(self, dout):
        for layer in reversed(self.layers):
            dout = layer.backward(dout)
        return dout
    
    def __call__(self, x):
        return self.forward(x)
    
    def train(self):
        for layer in self.layers:
            layer.train()
    
    def eval(self):
        for layer in self.layers:
            layer.eval()
    
    def parameters(self):
        """Return alle lagen met parameters."""
        return [layer for layer in self.layers if layer.params]

## 10.9 Alles Samen: MNIST Classifier

Nu testen we onze library op een echt probleem: handgeschreven cijfers herkennen (MNIST). We laden de dataset en normaliseren de pixelwaarden naar $[0, 1]$.

In [ ]:
# Laad MNIST
from sklearn.datasets import fetch_openml

print("MNIST laden...")
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X, y = mnist.data / 255.0, mnist.target.astype(int)

# Train/test split
X_train, X_test = X[:60000], X[60000:]
y_train, y_test = y[:60000], y[60000:]

print(f"Training: {X_train.shape}, Test: {X_test.shape}")

We bouwen een netwerk met drie Linear-lagen (les 2-4), ReLU-activaties (les 5), en He-initialisatie (les 9). De output-laag heeft 10 neuronen (één per cijfer) zonder activatie — de softmax zit ingebouwd in de `CrossEntropyLoss`. We optimaliseren met SGD met momentum (les 6).

In [ ]:
# Bouw het netwerk
model = Sequential([
    Linear(784, 256),
    ReLU(),
    Linear(256, 128),
    ReLU(),
    Linear(128, 10)
])

# Loss en optimizer
criterion = CrossEntropyLoss()
optimizer = SGD(model.parameters(), lr=0.1, momentum=0.9)

print("Model architectuur: 784 → 256 → ReLU → 128 → ReLU → 10")
print(f"Aantal lagen: {len(model.layers)}")

### Training loop

De training loop brengt alle concepten samen in één cyclus die we steeds herhalen:
1. **Forward pass** (les 2-4): input door het netwerk sturen
2. **Loss berekening**: cross-entropy meten hoe goed de voorspellingen zijn
3. **Backward pass** (les 7): gradiënten berekenen via backpropagation en de kettingregel
4. **Parameter update** (les 6): gradient descent past de weights aan

We trainen in **mini-batches** van 128 samples — een compromis tussen de stabiliteit van volledige gradient descent en de snelheid van stochastic gradient descent (les 6).

In [ ]:
def accuracy(model, X, y):
    """Bereken accuracy."""
    model.eval()
    logits = model(X)
    preds = np.argmax(logits, axis=1)
    model.train()
    return np.mean(preds == y)

# Training loop
batch_size = 128
n_epochs = 10
n_batches = len(X_train) // batch_size

train_losses = []
test_accs = []

print("Training starten...\n")

for epoch in range(n_epochs):
    model.train()
    
    # Shuffle
    idx = np.random.permutation(len(X_train))
    X_shuffled = X_train[idx]
    y_shuffled = y_train[idx]
    
    epoch_loss = 0
    
    for batch in range(n_batches):
        start = batch * batch_size
        end = start + batch_size
        
        X_batch = X_shuffled[start:end]
        y_batch = y_shuffled[start:end]
        
        # Forward
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        epoch_loss += loss
        
        # Backward
        optimizer.zero_grad()
        dout = criterion.backward()
        model.backward(dout)
        
        # Update
        optimizer.step()
    
    # Evalueer
    avg_loss = epoch_loss / n_batches
    test_acc = accuracy(model, X_test, y_test)
    
    train_losses.append(avg_loss)
    test_accs.append(test_acc)
    
    print(f"Epoch {epoch+1:2d}: Loss = {avg_loss:.4f}, Test Acc = {test_acc:.4f}")

print(f"\nFinale test accuracy: {test_accs[-1]*100:.2f}%")

De grafieken hieronder tonen het verloop van de training. Links de cross-entropy loss die moet dalen, rechts de test accuracy die moet stijgen. Een dalende loss met stijgende accuracy bevestigt dat het model leert om de likelihood van de correcte labels te maximaliseren.

In [ ]:
# Visualiseer training
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, 'b-', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(test_accs, 'r-', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Test Accuracy')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Tot slot bekijken we enkele voorspellingen op de testset. Het model produceert logits die via softmax (les 8) worden omgezet naar kansen — de klasse met de hoogste kans is de voorspelling. Groene titels zijn correcte voorspellingen, rode zijn fouten.

In [ ]:
# Visualiseer voorspellingen
model.eval()

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
indices = np.random.choice(len(X_test), 10, replace=False)

for ax, idx in zip(axes.flatten(), indices):
    img = X_test[idx].reshape(28, 28)
    logits = model(X_test[idx:idx+1])
    pred = np.argmax(logits)
    true = y_test[idx]
    
    ax.imshow(img, cmap='gray')
    color = 'green' if pred == true else 'red'
    ax.set_title(f'Pred: {pred}, True: {true}', color=color)
    ax.axis('off')

plt.suptitle('Model Voorspellingen', fontsize=14)
plt.tight_layout()
plt.show()

## 10.10 Samenvatting

We hebben een complete neural network library gebouwd met:

| Component | Wiskundige basis | Les |
|-----------|------------------|-----|
| Linear layer | Matrixvermenigvuldiging | 2-4 |
| Activations | Niet-lineaire functies + afgeleiden | 5 |
| Weight initialisatie | Variantie (He/Xavier) | 9 |
| Loss functions | Cross-entropy, MSE | 9 |
| Optimizers | Gradient descent + momentum | 6 |
| Backpropagation | Kettingregel | 7 |

**Alle wiskunde uit deze cursus komt samen in een werkend neuraal netwerk!**